# Nigeria Climate Data Analysis

## Purpose
This notebook focuses specifically on Nigeria's climate patterns as part of the regional analysis strategy. 
**Why we're analyzing Nigeria separately:**
- Nigeria has unique coastal and inland climate variations
- Regional temperature anomalies need isolated analysis
- Prevents skewed averages in multi-country datasets
- Enables targeted climate policy recommendations for Nigeria

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Set style for professional visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## Data Loading - Nigeria Specific

**Intent:** Load only Nigeria data to ensure focused analysis without cross-country contamination.

In [ ]:
# Load Nigeria-specific climate data
df_nigeria = pd.read_csv('../data/nigeria.csv')

# Filter to ensure we only have Nigeria data (quality check)
print(f"Dataset shape: {df_nigeria.shape}")
print(f"Date range: {df_nigeria['YEAR'].min()} to {df_nigeria['YEAR'].max()}")
print(f"Total records: {len(df_nigeria)}")

In [ ]:
# Data profiling for Nigeria
print("=== NIGERIA CLIMATE DATA PROFILING ===")
print("\nData Info:")
df_nigeria.info()

print("\nDescriptive Statistics:")
print(df_nigeria[['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M']].describe())

## Data Cleaning - Nigeria Specific

**Why we're cleaning Nigeria data separately:**
- Nigeria's coastal regions may have different outlier patterns
- Regional climate variations require country-specific thresholds
- Ensures Nigeria's climate patterns aren't masked by other countries

In [ ]:
# Check for missing values in Nigeria data
print("Missing values in Nigeria dataset:")
missing_values = df_nigeria.isnull().sum()
print(missing_values[missing_values > 0])

# Check for invalid values (-999) common in climate data
invalid_counts = {}
for col in ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M']:
    invalid_count = (df_nigeria[col] == -999).sum()
    if invalid_count > 0:
        invalid_counts[col] = invalid_count

print(f"\nInvalid values (-999) in Nigeria data: {invalid_counts}")

In [ ]:
# Clean Nigeria data
df_nigeria_clean = df_nigeria.copy()

# Replace invalid values with NaN
climate_columns = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M']
for col in climate_columns:
    df_nigeria_clean[col] = df_nigeria_clean[col].replace(-999, np.nan)

# Remove rows with critical missing data
initial_count = len(df_nigeria_clean)
df_nigeria_clean = df_nigeria_clean.dropna(subset=['T2M', 'PRECTOTCORR'])
final_count = len(df_nigeria_clean)

print(f"Nigeria data cleaning results:")
print(f"Initial records: {initial_count}")
print(f"After removing invalid/critical missing data: {final_count}")
print(f"Records removed: {initial_count - final_count} ({((initial_count - final_count)/initial_count)*100:.2f}%)")

## Nigeria Temperature Analysis

**Focus:** Understanding Nigeria's temperature patterns for agricultural and health policy planning.

In [ ]:
# Nigeria temperature trends
plt.figure(figsize=(15, 10))

# Create date column for time series
df_nigeria_clean['Date'] = pd.to_datetime(df_nigeria_clean['YEAR'].astype(str) + '-' + 
                                        df_nigeria_clean['DOY'].astype(str), format='%Y-%j')

# Plot 1: Daily Temperature Trends
plt.subplot(2, 2, 1)
plt.plot(df_nigeria_clean['Date'], df_nigeria_clean['T2M'], alpha=0.6, color='navy')
plt.title('Nigeria Daily Temperature Trends (2015-2026)')
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.xticks(rotation=45)

# Plot 2: Monthly Temperature Distribution
plt.subplot(2, 2, 2)
monthly_temp = df_nigeria_clean.groupby(df_nigeria_clean['Date'].dt.month)['T2M'].mean()
monthly_temp.plot(kind='bar', color='orange')
plt.title('Nigeria Average Monthly Temperature')
plt.xlabel('Month')
plt.ylabel('Temperature (°C)')
plt.xticks(range(12), ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                    'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], rotation=45)

# Plot 3: Temperature Range Analysis
plt.subplot(2, 2, 3)
temp_range = df_nigeria_clean['T2M_MAX'] - df_nigeria_clean['T2M_MIN']
plt.hist(temp_range.dropna(), bins=30, color='green', alpha=0.7)
plt.title('Nigeria Daily Temperature Range Distribution')
plt.xlabel('Temperature Range (°C)')
plt.ylabel('Frequency')

# Plot 4: Yearly Temperature Trends
plt.subplot(2, 2, 4)
yearly_temp = df_nigeria_clean.groupby('YEAR')['T2M'].mean()
plt.plot(yearly_temp.index, yearly_temp.values, marker='o', color='red')
plt.title('Nigeria Yearly Average Temperature')
plt.xlabel('Year')
plt.ylabel('Temperature (°C)')

plt.tight_layout()
plt.show()

# Print key statistics
print("=== NIGERIA TEMPERATURE ANALYSIS ===")
print(f"Average Temperature: {df_nigeria_clean['T2M'].mean():.2f}°C")
print(f"Max Temperature: {df_nigeria_clean['T2M_MAX'].max():.2f}°C")
print(f"Min Temperature: {df_nigeria_clean['T2M_MIN'].min():.2f}°C")
print(f"Average Daily Range: {temp_range.mean():.2f}°C")

## Nigeria Rainfall Analysis

**Purpose:** Analyze Nigeria's rainfall patterns crucial for agricultural planning and water resource management.

In [ ]:
# Nigeria rainfall analysis
plt.figure(figsize=(15, 10))

# Plot 1: Daily Rainfall
plt.subplot(2, 2, 1)
plt.plot(df_nigeria_clean['Date'], df_nigeria_clean['PRECTOTCORR'], alpha=0.6, color='blue')
plt.title('Nigeria Daily Rainfall (2015-2026)')
plt.xlabel('Date')
plt.ylabel('Rainfall (mm/day)')
plt.xticks(rotation=45)

# Plot 2: Monthly Rainfall Patterns
plt.subplot(2, 2, 2)
monthly_rain = df_nigeria_clean.groupby(df_nigeria_clean['Date'].dt.month)['PRECTOTCORR'].mean()
monthly_rain.plot(kind='bar', color='cyan')
plt.title('Nigeria Average Monthly Rainfall')
plt.xlabel('Month')
plt.ylabel('Rainfall (mm/day)')
plt.xticks(range(12), ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                    'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], rotation=45)

# Plot 3: Rainfall Distribution
plt.subplot(2, 2, 3)
plt.hist(df_nigeria_clean['PRECTOTCORR'].dropna(), bins=50, color='purple', alpha=0.7)
plt.title('Nigeria Daily Rainfall Distribution')
plt.xlabel('Rainfall (mm/day)')
plt.ylabel('Frequency')
plt.yscale('log')  # Log scale due to many zero rainfall days

# Plot 4: Yearly Rainfall Totals
plt.subplot(2, 2, 4)
yearly_rain = df_nigeria_clean.groupby('YEAR')['PRECTOTCORR'].sum()
plt.plot(yearly_rain.index, yearly_rain.values, marker='s', color='darkblue')
plt.title('Nigeria Yearly Total Rainfall')
plt.xlabel('Year')
plt.ylabel('Total Rainfall (mm/year)')

plt.tight_layout()
plt.show()

# Print key statistics
print("=== NIGERIA RAINFALL ANALYSIS ===")
print(f"Average Daily Rainfall: {df_nigeria_clean['PRECTOTCORR'].mean():.2f} mm/day")
print(f"Max Daily Rainfall: {df_nigeria_clean['PRECTOTCORR'].max():.2f} mm/day")
print(f"Rainy Days (>0.1mm): {(df_nigeria_clean['PRECTOTCORR'] > 0.1).sum()}")
print(f"Dry Days: {(df_nigeria_clean['PRECTOTCORR'] <= 0.1).sum()}")
print(f"Average Annual Rainfall: {yearly_rain.mean():.2f} mm/year")

## Nigeria Climate Anomalies Detection

**Why this matters for Nigeria:**
- Coastal Nigeria may experience different anomaly patterns
- Early detection supports climate adaptation planning
- Regional specificity improves policy relevance

In [ ]:
# Detect temperature anomalies in Nigeria
temp_mean = df_nigeria_clean['T2M'].mean()
temp_std = df_nigeria_clean['T2M'].std()

# Define anomalies (±2 standard deviations)
temp_upper_threshold = temp_mean + 2 * temp_std
temp_lower_threshold = temp_mean - 2 * temp_std

hot_days = df_nigeria_clean[df_nigeria_clean['T2M'] > temp_upper_threshold]
cold_days = df_nigeria_clean[df_nigeria_clean['T2M'] < temp_lower_threshold]

print("=== NIGERIA TEMPERATURE ANOMALIES ===")
print(f"Temperature Mean: {temp_mean:.2f}°C")
print(f"Temperature Std Dev: {temp_std:.2f}°C")
print(f"Upper Threshold: {temp_upper_threshold:.2f}°C")
print(f"Lower Threshold: {temp_lower_threshold:.2f}°C")
print(f"\nHot Days (>2σ): {len(hot_days)} ({len(hot_days)/len(df_nigeria_clean)*100:.2f}%)")
print(f"Cold Days (<-2σ): {len(cold_days)} ({len(cold_days)/len(df_nigeria_clean)*100:.2f}%)")

# Detect rainfall anomalies
rain_mean = df_nigeria_clean['PRECTOTCORR'].mean()
rain_std = df_nigeria_clean['PRECTOTCORR'].std()

heavy_rain_threshold = rain_mean + 2 * rain_std
heavy_rain_days = df_nigeria_clean[df_nigeria_clean['PRECTOTCORR'] > heavy_rain_threshold]

print(f"\n=== NIGERIA RAINFALL ANOMALIES ===")
print(f"Rainfall Mean: {rain_mean:.2f} mm/day")
print(f"Heavy Rain Threshold (>2σ): {heavy_rain_threshold:.2f} mm/day")
print(f"Heavy Rain Days: {len(heavy_rain_days)} ({len(heavy_rain_days)/len(df_nigeria_clean)*100:.2f}%)")

## Nigeria Climate Summary & Policy Insights

**Key findings for Nigeria's climate policy planning:**

In [ ]:
# Generate Nigeria-specific climate summary
nigeria_summary = {
    'temperature_profile': {
        'mean_temp': df_nigeria_clean['T2M'].mean(),
        'temp_variability': df_nigeria_clean['T2M'].std(),
        'extreme_hot_days': len(hot_days),
        'extreme_cold_days': len(cold_days)
    },
    'rainfall_profile': {
        'mean_daily_rainfall': df_nigeria_clean['PRECTOTCORR'].mean(),
        'annual_total_rainfall': yearly_rain.mean(),
        'rainy_season_months': monthly_rain[monthly_rain > monthly_rain.mean()].index.tolist(),
        'heavy_rain_events': len(heavy_rain_days)
    },
    'data_quality': {
        'total_records': len(df_nigeria_clean),
        'data_completeness': (1 - df_nigeria_clean.isnull().sum().sum() / (len(df_nigeria_clean) * len(climate_columns))) * 100,
        'years_covered': df_nigeria_clean['YEAR'].nunique()
    }
}

print("=== NIGERIA CLIMATE SUMMARY FOR POLICY MAKERS ===")
for category, metrics in nigeria_summary.items():
    print(f"\n{category.upper().replace('_', ' ')}:")
    for metric, value in metrics.items():
        if isinstance(value, float):
            print(f"  {metric.replace('_', ' ').title()}: {value:.2f}")
        else:
            print(f"  {metric.replace('_', ' ').title()}: {value}")

# Policy recommendations for Nigeria
print("\n=== NIGERIA-SPECIFIC POLICY RECOMMENDATIONS ===")
if nigeria_summary['temperature_profile']['extreme_hot_days'] > len(df_nigeria_clean) * 0.05:
    print("⚠️ HIGH TEMPERATURE RISK: Implement heat wave early warning systems")
if nigeria_summary['rainfall_profile']['annual_total_rainfall'] < 1000:
    print("⚠️ WATER SCARCITY RISK: Invest in water harvesting and conservation")
if nigeria_summary['rainfall_profile']['heavy_rain_events'] > len(df_nigeria_clean) * 0.1:
    print("⚠️ FLOOD RISK: Develop flood management infrastructure")

print("\n✅ Nigeria climate analysis completed successfully!")